# M06 — 多智能體系統

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

你會練到：
- 把 `create_agent` 包成大圖的一個 node（subgraph 概念）
- 用 `MessagesState` 當共享白板，讓上下文在 agent 間流動
- 寫一個 supervisor，用 `Command(goto=...)` 做分派與交棒（handoff）
- 印 ASCII 圖看多 agent 結構
- 🧪 練習：加入第三個專家

注意：本 notebook 仰賴你已在 repo 根目錄設好 `.env`（見第一冊 M00）。
程式碼不會被自動執行；註解中的「Expected output」幫你理解預期結果。

## 1. 環境準備

載入課程共用 helper，透過 `get_model()` 拿到一顆「供應商無關」的模型。
換 OpenAI / Anthropic / Ollama 只需改環境變數，這格不用動。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 先準備專家用的工具

多 agent 的重點是「各司其職」，所以先給每個專家**只屬於它的工具**。
這裡用兩個極簡的假工具，避免依賴外部服務：
- `web_search`：研究員用來「查資料」（這裡回傳寫死的字串，重點在流程不在真搜尋）。
- `word_count`：寫作者用來檢查字數。

工具一律從 `langchain.tools` 匯入 `tool`（不要用舊版 `from langchain.agents import tool`）。

In [ ]:
from langchain.tools import tool


@tool
def web_search(query: str) -> str:
    """Search the web and return a short factual snippet for the query."""
    # Fake search result so the lab runs without network or API keys.
    return f"（查詢『{query}』的結果）LangGraph 是有狀態的多智能體編排框架，2026 年為 v1.x。"


@tool
def word_count(text: str) -> int:
    """Return the number of whitespace-separated words in the text."""
    return len(text.split())

## 3. 建兩個專家 agent

用第一冊的 `create_agent` 各做一個專家。注意它們的 **system_prompt 互不干擾**：
- `research_agent`：只負責查證、整理事實，不負責文筆。
- `writing_agent`：只負責把素材寫成通順段落，不負責查資料。

這正是拆多 agent 的價值——兩種「人格」分開，各自的 prompt 都能寫得很專注。

In [ ]:
from langchain.agents import create_agent

research_agent = create_agent(
    model,
    tools=[web_search],
    system_prompt="你是嚴謹的研究員。只負責用工具查證並條列事實，不要寫成文章。",
)

writing_agent = create_agent(
    model,
    tools=[word_count],
    system_prompt="你是文筆流暢的寫作者。把前面研究員整理的事實，寫成一段通順的繁體中文短文。",
)

## 4. 把 agent 包成 node（subgraph）

`create_agent` 回傳的東西本身就是一張可 `invoke` 的圖，介面是吃 `{"messages": [...]}`、
回 `{"messages": [...]}`。我們把它包成大圖的一個 node：讀共享白板的 `messages` 餵進去，
再把**最後一則**訊息寫回白板。

⚠️ 只回傳 `result["messages"][-1]`，不要回整包——否則配 `add_messages` reducer
會把整段歷史重複塞回去（見 README 常見陷阱）。

In [ ]:
from langgraph.graph import MessagesState


def research_node(state: MessagesState) -> dict:
    # Run the research subgraph over the shared conversation.
    result = research_agent.invoke({"messages": state["messages"]})
    # Push only the agent's final message back to the shared state.
    return {"messages": [result["messages"][-1]]}


def writing_node(state: MessagesState) -> dict:
    result = writing_agent.invoke({"messages": state["messages"]})
    return {"messages": [result["messages"][-1]]}

## 5. supervisor：決定下一棒給誰

supervisor 是一個普通 node，讀共享狀態、判斷進度、用 `Command(goto=...)` 交棒。
為了讓流程**好懂又可預測**，這裡用一個簡單規則：
- 還沒研究過 → 先去 `research`。
- 研究過、還沒寫 → 去 `writing`。
- 都做完 → `goto=END` 收工。

我們在 state 裡塞一個 `next` 字串只是為了示範；判斷依據是「白板上累積了幾則
AI 訊息」。`Command` 一步同時「指定下一個 node」（這就是 handoff）。

真實專案常改成「讓一顆 LLM 讀對話後輸出下一棒」，原理一樣，只是把規則換成模型判斷。

In [ ]:
from langgraph.graph import END
from langgraph.types import Command
from langchain.messages import AIMessage


def supervisor(state: MessagesState) -> Command:
    # Count how many agent (AI) turns have happened so far.
    ai_turns = sum(1 for m in state["messages"] if isinstance(m, AIMessage))
    if ai_turns == 0:
        # Nothing done yet -> let the researcher gather facts first.
        return Command(goto="research")
    if ai_turns == 1:
        # Facts are on the board -> hand off to the writer.
        return Command(goto="writing")
    # Research + writing both done -> finish.
    return Command(goto=END)

## 6. 組裝多 agent 圖

結構：`START → supervisor`，supervisor 分派到 `research` 或 `writing`，
每個專家做完都**回到 supervisor** 再判斷下一步。共享狀態用內建的 `MessagesState`
（已含 `add_messages` reducer，多個 agent 的產出會累加而非互相覆蓋）。

因為 supervisor 用 `Command(goto=...)` 自己決定去向，它的「出口」要用
`add_conditional_edges` 把所有可能目標列出來，LangGraph 才知道這個 node 可能跳去哪。

In [ ]:
from langgraph.graph import StateGraph, START

builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor)
builder.add_node("research", research_node)
builder.add_node("writing", writing_node)

builder.add_edge(START, "supervisor")
# supervisor uses Command(goto=...); declare its possible destinations.
builder.add_conditional_edges(
    "supervisor",
    lambda state: state,  # routing is decided inside supervisor via Command
    ["research", "writing", END],
)
# Every expert reports back to the supervisor for the next decision.
builder.add_edge("research", "supervisor")
builder.add_edge("writing", "supervisor")

graph = builder.compile()

## 7. 印出 ASCII 圖

用文字圖確認多 agent 結構：supervisor 在中央，兩個專家在旁邊，且都有回頭的邊。
預期會看到 `supervisor`、`research`、`writing` 三個節點與彼此的連線。

In [ ]:
print(graph.get_graph().draw_ascii())
# Expected output (示意，實際排版依版本略有不同):
#         +-----------+
#         |  __start__ |
#         +-----------+
#               |
#               v
#        +--------------+
#        |  supervisor  | <---------+-----------+
#        +--------------+           |           |
#           |        |              |           |
#           v        v              |           |
#     +----------+ +---------+      |           |
#     | research |-+ writing |------+           |
#     +----------+   +-------+                  |
#                                            (END)

## 8. 跑一次完整流程

丟一個任務進去，看 supervisor 如何把它依序交給研究員、再交給寫作者，最後收工。
用 `stream(..., stream_mode="updates")` 可以一格一格看每個 node 的輸出
（M07 會深入串流）。

In [ ]:
task = {"messages": [{"role": "user", "content": "幫我寫一段介紹 LangGraph 是什麼的短文"}]}

for step in graph.stream(task, stream_mode="updates"):
    # Each step is {node_name: {"messages": [...]}}; show who ran.
    for node_name, update in step.items():
        last = update["messages"][-1] if update.get("messages") else None
        preview = (last.content[:40] + "...") if last and last.content else "(handoff)"
        print(f"[{node_name}] {preview}")
# Expected output (示意):
# [supervisor] (handoff)
# [research]  （查詢『LangGraph』的結果）LangGraph 是有狀態的...
# [supervisor] (handoff)
# [writing]   LangGraph 是一套有狀態的多智能體編排框架...
# [supervisor] (handoff)

## 9. 取最終結果

跑完後，最終短文就是白板上最後一則訊息。`invoke` 一次拿到完整結果。

In [ ]:
final_state = graph.invoke(task)
print(final_state["messages"][-1].content)
# Expected output: 一段由 writing_agent 寫出的、介紹 LangGraph 的繁體中文短文。

## 10. handoff 心智模型小結

觀察重點：研究員查到的素材**留在共享 `messages` 上**，寫作者一上場就讀得到——
你完全沒有手動傳參數，上下文是靠共享狀態流動的。這就是 handoff 的本質：
交棒不是「把資料打包傳過去」，而是「大家共用同一塊白板，換人來寫」。

補充概念（本 lab 用不到，但要知道）：若某個專家是**子圖**，而它想直接決定
「回父圖的哪個 node」，可在子圖內回傳：

    Command(goto="supervisor", graph=Command.PARENT)

沒加 `graph=Command.PARENT` 時，`goto` 只在當前圖內跳轉。

## 🧪 練習：加入第三個專家（校稿 agent）

現在的流程是 研究 → 寫作 → 結束。請加入第三個專家 `proofread_agent`，
讓流程變成 研究 → 寫作 → 校稿 → 結束。步驟：

1. 用 `create_agent` 做一個 `proofread_agent`，system_prompt 設成
   「你是校稿員，只修正錯字與語病，不改變原意」。
2. 仿照 `research_node` / `writing_node`，包一個 `proofread_node`。
3. 改 `supervisor`：把 `ai_turns == 2` 導去 `"proofread"`，`>= 3` 才 `goto=END`。
4. 在圖裡 `add_node("proofread", ...)`、把 `"proofread"` 加進 supervisor 的
   `add_conditional_edges` 目標清單，並 `add_edge("proofread", "supervisor")`。
5. 重新 `compile()`，印 ASCII 圖確認多了一個節點，再跑一次看三棒接力。

想一想：如果改成「讓 supervisor 用 LLM 讀對話來決定下一棒」，你的 supervisor
函式會怎麼改？（提示：用 `model.with_structured_output(...)` 讓模型輸出下一個
node 名稱。）

In [ ]:
# 在這裡寫你的答案（可參考下面骨架）：
#
# proofread_agent = create_agent(
#     model,
#     tools=[],
#     system_prompt="你是校稿員，只修正錯字與語病，不改變原意。",
# )
#
# def proofread_node(state: MessagesState) -> dict:
#     result = proofread_agent.invoke({"messages": state["messages"]})
#     return {"messages": [result["messages"][-1]]}
#
# def supervisor_v2(state: MessagesState) -> Command:
#     ai_turns = sum(1 for m in state["messages"] if isinstance(m, AIMessage))
#     if ai_turns == 0:
#         return Command(goto="research")
#     if ai_turns == 1:
#         return Command(goto="writing")
#     if ai_turns == 2:
#         return Command(goto="proofread")
#     return Command(goto=END)
#
# builder2 = StateGraph(MessagesState)
# builder2.add_node("supervisor", supervisor_v2)
# builder2.add_node("research", research_node)
# builder2.add_node("writing", writing_node)
# builder2.add_node("proofread", proofread_node)
# builder2.add_edge(START, "supervisor")
# builder2.add_conditional_edges(
#     "supervisor", lambda s: s, ["research", "writing", "proofread", END]
# )
# builder2.add_edge("research", "supervisor")
# builder2.add_edge("writing", "supervisor")
# builder2.add_edge("proofread", "supervisor")
# graph2 = builder2.compile()
# print(graph2.get_graph().draw_ascii())

## 小結 & 下一步

- 工具太多、職責太雜時，把單一 agent 拆成**各司其職的專家 agent**，每個只做一件事。
- 把 `create_agent` 包成 node 就是 **subgraph**；用 **supervisor** 做分派、用
  **`Command(goto=...)`** 做 handoff，靠**共享 `MessagesState`** 傳遞上下文。
- 這一切都是 M01（node）+ M02（reducer）+ M03（路由/Command）+ 第一冊
  （`create_agent`）的組合，不是新魔法。

**下一步：M07 — 串流、可觀測與整合專案**。我們會替這些圖加上串流輸出與觀測，
並把第二冊學到的東西收斂成一個完整專案。